# 04 — Product Performance Analysis

**Goal:** Identify top-selling and most profitable product categories.
Analyse average price, units sold, and revenue contribution.

---

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from analysis import set_style, top_categories, plot_top_categories, PALETTE

set_style()
print('Ready ✅')

In [ ]:
master = pd.read_csv('../data/processed/master.csv', parse_dates=['order_purchase_timestamp'])
print(f'Loaded {master.shape[0]:,} rows')

## 1. Top 10 Categories by Revenue

In [ ]:
cats = top_categories(master, n=10)
cats.style.format({'revenue': 'R${:,.2f}', 'avg_price': 'R${:,.2f}'})

In [ ]:
plot_top_categories(master, n=10, save_path='../images/top_categories.png')

## 2. Units Sold vs Revenue (Bubble Chart)

In [ ]:
cats20 = top_categories(master, n=20)

fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(
    cats20['units_sold'],
    cats20['revenue'],
    s=cats20['avg_price'] * 3,
    c=range(len(cats20)),
    cmap='tab20',
    alpha=0.75,
    edgecolors='white',
    linewidths=0.8
)

for _, row in cats20.iterrows():
    ax.annotate(row['category_en'],
                (row['units_sold'], row['revenue']),
                fontsize=7.5, ha='center', va='bottom')

ax.set_xlabel('Units Sold')
ax.set_ylabel('Revenue (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e3:.0f}k'))
ax.set_title('Units Sold vs Revenue by Category\n(bubble size = avg price)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/product_bubble.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Average Price by Category (Top 15)

In [ ]:
cats15 = top_categories(master, n=15).sort_values('avg_price')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(cats15['category_en'], cats15['avg_price'],
        color=PALETTE['accent'], alpha=0.85)
ax.set_title('Average Product Price by Category (Top 15 by Revenue)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Average Price (R$)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
plt.tight_layout()
plt.savefig('../images/avg_price_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Monthly Revenue by Top 5 Categories (Stacked Area)

In [ ]:
top5 = top_categories(master, n=5)['category_en'].tolist()
monthly_cat = (
    master[master['category_en'].isin(top5)]
    .groupby(['order_yearmonth', 'category_en'])['payment_value']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.stackplot(
    monthly_cat['order_yearmonth'].astype(str),
    [monthly_cat[cat] for cat in top5],
    labels=top5,
    alpha=0.75
)
ax.legend(loc='upper left', fontsize=8)
ax.set_title('Monthly Revenue — Top 5 Categories (Stacked)', fontsize=13, fontweight='bold')
ax.set_ylabel('Revenue (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1e3:.0f}k'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../images/top5_categories_trend.png', dpi=150, bbox_inches='tight')
plt.show()